# Разметка и сравнение train guide_v3

Char-LR обучен отдельно на guide_v2 и guide_v3 с одинаковым разбиением и порогом 0,50. Обе версии оценены на одном test_guide_v3. В общем результате исключены пересечения между тестовыми срезами. По train проверены кандидаты под обновлённые правила, спорные метки перечислены в журнале. Результаты относятся к внутренним тестам.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'dataset_review' / 'compare_guide_v3.py').exists())
sys.path.insert(0, str(ROOT / 'dataset_review'))

from compare_guide_v2 import fingerprint
from compare_guide_v3 import OUTPUT, TEST_OUT, TRAIN_OUT, VERSION_PATHS, compare

RERUN = False

if RERUN:
    result = compare()
else:
    result = json.loads(OUTPUT.read_text(encoding='utf-8'))

assert result['test_sha256'] == {
    name: fingerprint(TEST_OUT / name) for name in result['test_sha256']
}
assert all(result['runs'][version]['train_sha256'] == fingerprint(path)
           for version, path in VERSION_PATHS.items())
assert len({run['fit_item_ids_sha256'] for run in result['runs'].values()}) == 1

display(pd.DataFrame([
    {'Версия': version, 'Строк в fit': run['fit_rows'], 'Реальные': run['fit_by_origin']['real'],
     'Синтетические': run['fit_by_origin']['synthetic'],
     'Пересечения fit и теста': run['fit_eval_overlap']}
    for version, run in result['runs'].items()
]))
print('Повторных строк между тестовыми срезами:', result['duplicate_evaluation_rows'])
print('Dev-строк из stress, которые встречаются в полном train:',
      len(result['runs']['guide_v3']['train_full_stress_overlaps']))

## Сравнение по тестовым наборам

In [ ]:
slices = [
    'unique_all', 'gold', 'hard', 'stress_all',
    'gold_core', 'hard_empty', 'decoupled_solo_and_pairs',
    'context_traps_and_negations', 'dirty_text_and_homoglyphs',
    'non_101_source_mcid', 'mentor_borderline_switches',
]
old = result['runs']['guide_v2']['metrics']
new = result['runs']['guide_v3']['metrics']

def count_change(before, after):
    return f'{before} → {after}'

table = []
for name in slices:
    a, b = old[name], new[name]
    table.append({
        'Набор': name,
        'Строк': b['rows'],
        'F1 v2, %': round(a['micro_f1'] * 100, 2),
        'F1 v3, %': round(b['micro_f1'] * 100, 2),
        'Δ F1, п.п.': round((b['micro_f1'] - a['micro_f1']) * 100, 2),
        'Пропуски 106': count_change(a['per_class']['106']['fn'], b['per_class']['106']['fn']),
        'Ложные 106': count_change(a['per_class']['106']['fp'], b['per_class']['106']['fp']),
        'Полное совпадение, %': f"{a['exact_match']:.2%} → {b['exact_match']:.2%}",
    })
display(pd.DataFrame(table))

cross_source = new['non_101_source_mcid']['per_class']['101']
print(f"Категория 101 в non_101_source_mcid. "
      f"Найдено {cross_source['tp']}, пропущено {cross_source['fn']}. "
      "В train все исходные категории равны 101.")

## Ошибки по категориям на 475 уникальных объявлениях

In [ ]:
a, b = old['unique_all']['per_class'], new['unique_all']['per_class']
display(pd.DataFrame([
    {'Категория': category,
     'Верные': count_change(a[category]['tp'], b[category]['tp']),
     'Ложные': count_change(a[category]['fp'], b[category]['fp']),
     'Пропуски': count_change(a[category]['fn'], b[category]['fn'])}
    for category in b
]))

## Изменения и спорные примеры

Журналы содержат исходный ID, прежние и новые наборы меток, цитату и причину. Копии gold в stress получают те же метки.

In [ ]:
train_changes = pd.read_csv(TRAIN_OUT / 'label_changes.csv')
test_changes = pd.read_csv(TEST_OUT / 'label_changes.csv')
train_changes['Тип источника'] = train_changes['synth_task_id'].notna().map({True: 'синтетика', False: 'реальные'})
display(pd.crosstab(train_changes['Тип источника'], train_changes['new'])
        .rename(columns={0: 'Снято', 1: 'Добавлено'}))
columns = ['dataset', 'row_id', 'mcId', 'old_labels', 'new_labels', 'quote', 'reason']
with pd.option_context('display.max_colwidth', None):
    display(pd.crosstab(test_changes['dataset'], test_changes['new'])
            .rename(columns={0: 'Снято', 1: 'Добавлено'}))
    display(test_changes[columns])
    display(pd.concat([pd.read_csv(p / 'ambiguous.csv')[columns] for p in (TEST_OUT, TRAIN_OUT)],
                      ignore_index=True))